In [5]:
from osgeo import gdal, gdalconst
from osgeo import ogr, osr

fn = r"C:\Users\tanzhenyu\Dataware\GeoPy\chn_adm_ocha_2020_shp\chn_admbnda_adm1_ocha_2020.shp"
# 使用gdal.OpenEx()函数返回的是gdal.Dataset，该函数需要两个参数，第二个参数可以指定打开的是栅格数据还是矢量数据
ds: gdal.Dataset = gdal.OpenEx(fn, gdalconst.OF_VECTOR)

# 一个Dataset中可以包含多个Layer
lyr: ogr.Layer = ds.GetLayerByName('chn_admbnda_adm1_ocha_2020')
srs: osr.SpatialReference = lyr.GetSpatialRef()
print(f'空间参考信息：{srs}')
# 一个Layer中包含多个Feature
defn: ogr.FeatureDefn = lyr.GetLayerDefn()
for i in range(defn.GetFieldCount()):
    fd: ogr.FieldDefn = defn.GetFieldDefn(i)
    print(f'字段名称：{fd.GetName()}\t字段类型：{fd.GetTypeName()}')

# Feature由几何对象（Geometry）和属性字段（Field）组成
feat: ogr.Feature = lyr.GetFeature(0)
geom: ogr.Geometry = feat.GetGeometryRef()
# 将该几何对象以OGC WKT格式进行输出
print(geom.ExportToWkt())
print('FID为0的要素的属性值：')
for i in range(feat.GetFieldCount()):
    print(feat.GetField(i))

del ds

空间参考信息：GEOGCS["WGS 84",
    DATUM["WGS_1984",
        SPHEROID["WGS 84",6378137,298.257223563,
            AUTHORITY["EPSG","7030"]],
        AUTHORITY["EPSG","6326"]],
    PRIMEM["Greenwich",0,
        AUTHORITY["EPSG","8901"]],
    UNIT["degree",0.0174532925199433,
        AUTHORITY["EPSG","9122"]],
    AXIS["Latitude",NORTH],
    AXIS["Longitude",EAST],
    AUTHORITY["EPSG","4326"]]
字段名称：ADM1_EN	字段类型：String
字段名称：ADM1_ZH	字段类型：String
字段名称：ADM1_PCODE	字段类型：String
字段名称：ADM0_EN	字段类型：String
字段名称：ADM0_ZH	字段类型：String
字段名称：ADM0_PCODE	字段类型：String
POLYGON ((109.434530388 33.1559144340001,109.434165852 33.1539540100001,109.461935767 33.124187492,109.503335606 33.122000869,109.572996049 33.1218744060001,109.636403546 33.1048275380001,109.694411341 33.0832696620001,109.76116998 33.056456537,109.735739453 33.0128546920001,109.74955613 32.9607597790001,109.730345217 32.9384541220001,109.731068383 32.890657773,109.758275479 32.8713954850001,109.784301501 32.8521424950001,109.806520864 32.852178489,10

In [12]:
from osgeo import ogr

fn = r"C:\Users\TheOne\Dataware\GeoPy\chn_adm_ocha_2020_shp\chn_admbnda_adm1_ocha_2020.shp"
# 使用ogr.Open()函数返回的是ogr.DataSource
ds: ogr.DataSource = ogr.Open(fn)
lyr: ogr.Layer = ds.GetLayerByName('chn_admbnda_adm1_ocha_2020')
defn: ogr.FeatureDefn = lyr.GetLayerDefn()
for i in range(defn.GetFieldCount()):
    fd: ogr.FieldDefn = defn.GetFieldDefn(i)
    print(f'字段名称：{fd.GetName()}\t字段类型：{fd.GetTypeName()}')

feat: ogr.Feature = lyr.GetFeature(1)
print('FID为1的要素的属性值：')
for i in range(feat.GetFieldCount()):
    print(feat.GetField(i))

del ds

字段名称：ADM1_EN	字段类型：String
字段名称：ADM1_ZH	字段类型：String
字段名称：ADM1_PCODE	字段类型：String
字段名称：ADM0_EN	字段类型：String
字段名称：ADM0_ZH	字段类型：String
字段名称：ADM0_PCODE	字段类型：String
FID为1的要素的属性值：
Shanghai Municipality
上海市
CN031
China
中国
CN


In [1]:
from osgeo import ogr
from osgeo import osr
import json
import os
# 设置Shapefile文件中字符串编码，Windows默认的编码和系统一样是GBK
os.environ['SHAPE_ENCODING'] = 'GBK'

# 这里的GeoJSON文件使用的是UTF-8编码，打开文本文件时需要注意文件的编码格式
ifn = r"C:\Users\TheOne\Dataware\GeoPy\geojson-map-china\china.json"
with open(ifn, encoding='UTF-8') as f:
    china = json.load(f)

# 创建DataSource
ofn = r"C:\Users\TheOne\Dataware\GeoPy\geojson-map-china\china.shp"
driver: ogr.Driver = ogr.GetDriverByName('ESRI Shapefile')
ds: ogr.DataSource = driver.CreateDataSource(ofn)

# 创建WGS84空间参考（我们最熟悉的地理参考系统）
srs = osr.SpatialReference()
srs.ImportFromEPSG(4326)

# 创建图层
layer: ogr.Layer = ds.CreateLayer('province', srs, ogr.wkbPolygon)
# 添加属性定义
fname: ogr.FieldDefn = ogr.FieldDefn('Name', ogr.OFTString)
fname.SetWidth(24)
layer.CreateField(fname)
fcx = ogr.FieldDefn('CenterX', ogr.OFTReal)
layer.CreateField(fcx)
fcy = ogr.FieldDefn('CenterY', ogr.OFTReal)
layer.CreateField(fcy)

# 变量GeoJSON中的features
for f in china['features']:
    # 新建Feature并且给其属性赋值
    feature = ogr.Feature(layer.GetLayerDefn())
    feature.SetField('Name', f['properties']['name'])
    feature.SetField('CenterX', f['properties']['cp'][0])
    feature.SetField('CenterY', f['properties']['cp'][1])

    # 设置Feature的几何属性Geometry
    polygon = ogr.CreateGeometryFromJson(str(f['geometry']))
    feature.SetGeometry(polygon)
    # 创建Feature
    layer.CreateFeature(feature)
    del feature
ds.FlushCache()

del ds

In [5]:
import os
from osgeo import gdal, ogr
os.environ['SHAPE_ENCODING'] = 'GBK'

# 将Shapefile转为KML文件
ifn = r"C:\Users\TheOne\Dataware\GeoPy\geojson-map-china\china.shp"
ofn = r"C:\Users\TheOne\Dataware\GeoPy\geojson-map-china\china.kml"
gdal.VectorTranslate(ofn, ifn, format='KML')

# 将Shapefile转为GPKG文件
ofn = r"C:\Users\TheOne\Dataware\GeoPy\geojson-map-china\china.gpkg"
gdal.VectorTranslate(ofn, ifn, format='GPKG')
# 打开GPKG文件读取第一个要素
ds = ogr.Open(ofn)
lyr: ogr.Layer = ds.GetLayer(0)
defn: ogr.FeatureDefn = lyr.GetLayerDefn()
feat = ds.GetLayer(0).GetFeature(1)
for i in range(feat.GetFieldCount()):
    print(f'{defn.GetFieldDefn(i).GetName()}：{feat.GetField(i)}')

del ds

Name：新疆维吾尔自治区
CenterX：84.9023
CenterY：42.148


In [1]:
from osgeo import gdal, ogr

ifn = r"C:\Users\TheOne\Dataware\GeoPy\geojson-map-china\china.shp"
ofn = r"C:\Users\TheOne\Dataware\GeoPy\geojson-map-china\china_reprojected.shp"

# 使用命令行API转换
# 输出数据投影定义，参考资料：http://spatialreference.org/ref/sr-org/8657
# 使用"""表示多行字符串
srs_def = """+proj=aea +lat_1=25 +lat_2=47 +lat_0=30 +lon_0=105 +x_0=0 +y_0=0
+ellps=WGS84 +datum=WGS84 +units=m +no_defs """
gdal.VectorTranslate(ofn, ifn, dstSRS=srs_def, reproject=True)

ds = ogr.Open(ofn)
lyr = ds.GetLayer(0)
srs = lyr.GetSpatialRef()  # 输入数据投影
print(srs)
del ds

PROJCS["unknown",
    GEOGCS["GCS_unknown",
        DATUM["WGS_1984",
            SPHEROID["WGS 84",6378137,298.257223563,
                AUTHORITY["EPSG","7030"]],
            AUTHORITY["EPSG","6326"]],
        PRIMEM["Greenwich",0],
        UNIT["Degree",0.0174532925199433]],
    PROJECTION["Albers_Conic_Equal_Area"],
    PARAMETER["latitude_of_center",30],
    PARAMETER["longitude_of_center",105],
    PARAMETER["standard_parallel_1",25],
    PARAMETER["standard_parallel_2",47],
    PARAMETER["false_easting",0],
    PARAMETER["false_northing",0],
    UNIT["metre",1,
        AUTHORITY["EPSG","9001"]],
    AXIS["Easting",EAST],
    AXIS["Northing",NORTH]]


In [1]:
from osgeo import ogr, osr

src_fn = r"C:\Users\TheOne\Dataware\GeoPy\geojson-map-china\china.shp"
dst_fn = r"C:\Users\TheOne\Dataware\GeoPy\geojson-map-china\china_reprojected.shp"

src_ds = ogr.Open(src_fn)
src_layer = src_ds.GetLayer(0)
src_srs = src_layer.GetSpatialRef()  # 输入数据投影

# 输出数据投影定义，参考资料：http://spatialreference.org/ref/sr-org/8657
srs_def = """+proj=aea +lat_1=25 +lat_2=47 +lat_0=30 +lon_0=105 +x_0=0 +y_0=0
+ellps=WGS84 +datum=WGS84 +units=m +no_defs """
dst_srs = osr.SpatialReference()
dst_srs.ImportFromProj4(srs_def)

# 创建转换对象
ctx = osr.CoordinateTransformation(src_srs, dst_srs)

# 创建输出文件
driver = ogr.GetDriverByName('ESRI Shapefile')
dst_ds = driver.CreateDataSource(dst_fn)
dst_layer = dst_ds.CreateLayer('province', dst_srs, ogr.wkbPolygon)

# 给输出文件图层添加属性定义
layer_def = src_layer.GetLayerDefn()
for i in range(layer_def.GetFieldCount()):
    field_def = layer_def.GetFieldDefn(i)
    dst_layer.CreateField(field_def)

# 循环遍历源Shapefile中的几何体添加到目标文件中
src_feat = src_layer.GetNextFeature()
while src_feat:
    geometry = src_feat.GetGeometryRef()
    geometry.Transform(ctx)
    dst_feat = ogr.Feature(layer_def)
    dst_feat.SetGeometry(geometry)  # 设置Geometry
    # 依次设置属性值
    for i in range(layer_def.GetFieldCount()):
        field_def = layer_def.GetFieldDefn(i)
        field_name = field_def.GetName()
        dst_feat.SetField(field_name, src_feat.GetField(field_name))
    dst_layer.CreateFeature(dst_feat)
    src_feat = src_layer.GetNextFeature()
dst_ds.FlushCache()

del src_ds
del dst_ds

In [3]:
from osgeo import gdal

# 打开栅格数据集
fn = r"C:\Users\TheOne\Dataware\GeoPy\wsiearth.tif"
ds: gdal.Dataset = gdal.Open(fn)

# 获得栅格数据的一些重要信息
print(f'投影信息：{ds.GetSpatialRef()}')
print(f'栅格波段数：{ds.RasterCount}')
print(f'栅格列数（宽度）：{ds.RasterXSize}')
print(f'栅格行数（高度）：{ds.RasterYSize}')

# 获取数据集的元数据信息
metadata = ds.GetMetadata_Dict()
for key, value in metadata.items():
    print(f'{key} -> {value}')


for b in range(ds.RasterCount):
    # 注意GDAL中的band计数是从1开始的
    band: gdal.Band = ds.GetRasterBand(b + 1)
    # 波段数据的一些信息
    print(f'数据类型：{gdal.GetDataTypeName(band.DataType)}')  # DataType属性返回的是数字
    print(f'NoData值：{band.GetNoDataValue()}')  # 很多影像都是NoData，我们在做数据处理时要特别对待
    print(f'统计值（最大值最小值）：{band.ComputeRasterMinMax()}')  # 有些数据本身就存储了统计信息，有些数据没有需要计算

# 关闭数据集
del ds

投影信息：GEOGCS["WGS 84",
    DATUM["WGS_1984",
        SPHEROID["WGS 84",6378137,298.257223563,
            AUTHORITY["EPSG","7030"]],
        AUTHORITY["EPSG","6326"]],
    PRIMEM["Greenwich",0],
    UNIT["degree",0.0174532925199433,
        AUTHORITY["EPSG","9122"]],
    AXIS["Latitude",NORTH],
    AXIS["Longitude",EAST],
    AUTHORITY["EPSG","4326"]]
栅格波段数：1
栅格列数（宽度）：10020
栅格行数（高度）：5010
AREA_OR_POINT -> Area
TIFFTAG_RESOLUTIONUNIT -> 2 (pixels/inch)
TIFFTAG_SOFTWARE -> IMAGINE TIFF Support
Copyright 1991 - 1999 by ERDAS, Inc. All Rights Reserved
@(#)$RCSfile: etif.c $ $Revision: 1.9.1.2 $ $Date: 2001/12/05 00:33:12Z $
TIFFTAG_XRESOLUTION -> 1
TIFFTAG_YRESOLUTION -> 1
数据类型：Byte
NoData值：None
统计值（最大值最小值）：(0.0, 255.0)


In [ ]:
from osgeo import gdal

gdal.UseExceptions()

# 打开栅格数据集
fn = "/Users/tanzhenyu/Dataware/GeoPy/wsiearth.tif"
ds = gdal.Open(fn)
# 在数据集层面转换
image = ds.ReadAsArray()

print(f'数据的尺寸：{image.shape}')
# 输出结果为：数据的尺寸：(3, 4800, 4800)
# 这说明ReadAsArray方法将每个波段都转换为了一个二维数组

# 获得第一个波段的数据
band1 = image[0]

# 在波段层面的转换
for b in range(ds.RasterCount):
    # 注意GDAL中的band计数是从1开始的
    band = ds.GetRasterBand(b + 1)
    band = band.ReadAsArray()
    print(f'波段大小：{band.shape}')

# 关闭数据集
del ds

In [5]:
from osgeo import gdal_array
# 使用gdal_array模块
fn = r"C:\Users\TheOne\Dataware\GeoPy\wsiearth.tif"
data = gdal_array.LoadFile(fn)
print(f'数据的尺寸：{data.shape}')

数据的尺寸：(5010, 10020)
